In [104]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

import re #Regular Expression library to remove all html tags

import nltk #Natural language toolkit to remove stop words
# nltk.download('stopwords')
from nltk.corpus import stopwords

from nltk.stem.porter import PorterStemmer

#convert to tabular data
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB

from sklearn.metrics import accuracy_score

In [33]:
df = pd.read_csv('sentiment.csv')

In [34]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [35]:
df.shape

(50000, 2)

In [36]:
#One Review

df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

**Text Cleaning**

1) Sample 10000 rows
2) Remove HTML Tags
3) Remove Special Characters
4) Converting Everything to lower case
5) Removing Stop words
6) Stemming (e.g play, played, playing -> play)

In [37]:
df = df.sample(10000)

In [38]:
df.shape

(10000, 2)

In [39]:
df.isnull().sum() #No missing value

review       0
sentiment    0
dtype: int64

In [40]:
df.dtypes.value_counts()

object    2
Name: count, dtype: int64

In [41]:
#Encoding label

encoder = LabelEncoder()
df['sentiment'] = encoder.fit_transform(df['sentiment'])

In [42]:
df.dtypes.value_counts()

object    1
int64     1
Name: count, dtype: int64

In [43]:
#Removing HTML tags

def clean_html(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)

In [44]:
df['review'] = df['review'].apply(clean_html)

In [45]:
df.head()

,review,sentiment
13818,The interaction between Portman and Sarandon w...,1
40835,"""Edge of the City"" is another movie that owes ...",1
25825,Hollywood has made a lot of strange movies ove...,0
38256,"Alas, it seems that the golden times of stylis...",0
34590,An elite American military team which of cours...,0


In [46]:
#Converting text to lower case

def covert_lower(text):
    return text.lower()

In [47]:
df['review'] = df['review'].apply(covert_lower)

In [48]:
df.head()

,review,sentiment
13818,the interaction between portman and sarandon w...,1
40835,"""edge of the city"" is another movie that owes ...",1
25825,hollywood has made a lot of strange movies ove...,0
38256,"alas, it seems that the golden times of stylis...",0
34590,an elite american military team which of cours...,0


In [49]:
 #Remove special characters

def remove_special(text):
    x = ''

    for i in text:
        if i.isalnum(): #alpha numeric or not ?
            x+=i
        else:
            x+=' '
    return x

In [50]:
remove_special("testing@ removal%#functi@on")

'testing  removal  functi on'

In [51]:
#Remove the Stopwords
stop_words = set(stopwords.words('english'))

# len(stopwords.words('english'))
#stopwords.words('english')

In [52]:
def remove_stopwords(text):
    x = []
    for i in text.split():
        if i not in stop_words:
            x.append(i)
    return x

In [53]:
 df['review'] = df['review'].apply(remove_stopwords)

In [54]:
df.head()

,review,sentiment
13818,"[interaction, portman, sarandon, quite, intere...",1
40835,"[""edge, city"", another, movie, owes, lot, cred...",1
25825,"[hollywood, made, lot, strange, movies, years,...",0
38256,"[alas,, seems, golden, times, stylish, italian...",0
34590,"[elite, american, military, team, course, happ...",0


In [59]:
#Stemming

ps = PorterStemmer() #convert to base form, loving to love

In [60]:
def stem_words(text):
    y = []
    for i in text:
        y.append(ps.stem(i))
    return y

In [61]:
stem_words(['playing', 'played', 'ball'])

['play', 'play', 'ball']

In [62]:
 df['review'] = df['review'].apply(stem_words)

In [64]:
df.head()

,review,sentiment
13818,"[interact, portman, sarandon, quit, interestin...",1
40835,"[""edg, city"", anoth, movi, owe, lot, credit, ""...",1
25825,"[hollywood, made, lot, strang, movi, years,, n...",0
38256,"[alas,, seem, golden, time, stylish, italian, ...",0
34590,"[elit, american, militari, team, cours, happen...",0


In [66]:
def join_back(list_input):
    return " ".join(list_input)

In [67]:
 df['review'] = df['review'].apply(join_back)

In [68]:
df.head()

,review,sentiment
13818,"interact portman sarandon quit interesting, re...",1
40835,"""edg city"" anoth movi owe lot credit ""on water...",1
25825,"hollywood made lot strang movi years, none str...",0
38256,"alas, seem golden time stylish italian cinema ...",0
34590,elit american militari team cours happen inclu...,0


In [134]:
#Convert text to tabular data

cv = CountVectorizer()

In [135]:
X = cv.fit_transform(df['review']).toarray() #Convert to numpy array

In [142]:
X.shape

(10000, 50106)

In [143]:
X[0]

array([0, 0, 0, ..., 0, 0, 0], shape=(50106,))

In [144]:
y = df.iloc[:,-1].values #convert pd to numpy array
#:    → all rows
#-1   → last column

In [145]:
y

array([1, 1, 0, ..., 1, 1, 1], shape=(10000,))

In [146]:
y.shape

(10000,)

**Data now Ready**

**Splitting Data**

In [147]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20)

In [148]:
X_train.shape

(8000, 50106)

In [149]:
X_test.shape

(2000, 50106)

In [150]:
y_train.shape

(8000,)

In [151]:
y_test.shape

(2000,)

**Training Models**

In [152]:
model1 = MultinomialNB()
model2 = GaussianNB()
model3 = BernoulliNB()

In [153]:
model1.fit(X_train, y_train)
model2.fit(X_train, y_train)
model3.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,binarize,0.0
,fit_prior,True
,class_prior,None


In [154]:
y_pred1 = model1.predict(X_test)
y_pred2 = model2.predict(X_test)
y_pred3 = model3.predict(X_test)

In [155]:
print(f"Multinomial NB:  {(accuracy_score(y_test, y_pred1))*100}%")
print(f"Guassian NB:  {(accuracy_score(y_test, y_pred2))*100}%")
print(f"Bernoulli NB:  {(accuracy_score(y_test, y_pred3))*100}%")

Multinomial NB:  82.6%
Guassian NB:  63.349999999999994%
Bernoulli NB:  82.0%
